### Import needed libraries

In [30]:
import os
import pickle
import numpy as np
from sklearn.model_selection import train_test_split # type: ignore
from sklearn.preprocessing import LabelEncoder # type: ignore
from tensorflow.keras.models import Sequential # type: ignore
from tensorflow.keras.layers import Conv1D, Dense, Flatten, BatchNormalization, Dropout # type: ignore
from tensorflow.keras.utils import to_categorical # type: ignore
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import os

# Point XLA to your CUDA/libdevice directory
os.environ['XLA_FLAGS'] = '--xla_gpu_cuda_data_dir=/usr/lib/cuda'


### Load data

In [26]:
X = []
y = []

for file in os.listdir("pickles/"):
    if os.path.exists("pickles/" + file):
        with open("pickles/" + file, "rb") as f:
            data = pickle.load(f)
        for entry in data:
            points = entry["points"]  # points is a list of [x, y]
            if points:  # skip empty
                X.append(points)
                y.append(file.split(".")[0])
    else:
        print(f"{file} not found!")

max_points = max(len(sample) for sample in X)

X_padded = []
for sample in X:
    arr = np.array(sample, dtype=np.float32)
    if len(arr) < max_points:
        padding = np.zeros((max_points - len(arr), 2), dtype=np.float32)
        arr = np.vstack([arr, padding])
    X_padded.append(arr)

In [27]:
print(f"Total samples: {len(X_padded)}")
print(f"Max points: {max_points}")
print(f"Min points: {min(len(sample) for sample in X_padded)}")
print(f"Average points: {sum(len(sample) for sample in X_padded) / len(X_padded)}")
print(f"Unique classes: {len(set(y))}")
print(f"Class distribution: {dict(zip(set(y), [y.count(class_name) for class_name in set(y)]))}")
print(f"Sample shape: {X_padded[0].shape}")
print(f"Sample: {X_padded[1]}")
print(f"Label: {y[1]}")

Total samples: 3160
Max points: 234
Min points: 234
Average points: 234.0
Unique classes: 8
Class distribution: {'bueno': 620, 'cuanto': 340, 'nosign': 320, 'cuanto_precio': 280, 'donde': 320, 'que_tal': 360, 'tarde': 480, 'hola': 440}
Sample shape: (234, 2)
Sample: [[292. 205.]
 [294. 207.]
 [294. 207.]
 [298. 209.]
 [298. 209.]
 [303. 211.]
 [303. 211.]
 [310. 213.]
 [310. 213.]
 [317. 213.]
 [317. 213.]
 [325. 213.]
 [325. 213.]
 [332. 212.]
 [332. 212.]
 [339. 210.]
 [339. 210.]
 [343. 208.]
 [343. 208.]
 [345. 206.]
 [292. 205.]
 [293. 203.]
 [293. 203.]
 [296. 200.]
 [296. 200.]
 [301. 196.]
 [301. 196.]
 [309. 193.]
 [309. 193.]
 [317. 194.]
 [317. 194.]
 [325. 193.]
 [325. 193.]
 [334. 196.]
 [334. 196.]
 [339. 200.]
 [339. 200.]
 [344. 204.]
 [344. 204.]
 [345. 206.]
 [296. 204.]
 [299. 204.]
 [299. 204.]
 [301. 204.]
 [301. 204.]
 [306. 203.]
 [306. 203.]
 [311. 203.]
 [311. 203.]
 [317. 203.]
 [317. 203.]
 [324. 203.]
 [324. 203.]
 [330. 204.]
 [330. 204.]
 [335. 204.]
 [335

In [ ]:
X_array = np.array(X_padded)
X_array = X_array.reshape(X_array.shape[0], X_array.shape[1], 2)

print(f"Input shape: {X_array.shape}")

le = LabelEncoder()
y_encoded = le.fit_transform(y)
y_categorical = to_categorical(y_encoded)

X_train, X_test, y_train, y_test = train_test_split(
    X_array, y_categorical, test_size=0.2, random_state=42, stratify=y_categorical
)

print(f"Training samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")

model = Sequential([
    Conv1D(64, kernel_size=3, activation='relu', input_shape=(X_array.shape[1], 2)),
    Conv1D(64, kernel_size=3, activation='relu'),
    Dropout(0.3),
    Flatten(),
    Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
    Dropout(0.5),
    Dense(len(le.classes_), activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau # type: ignore

callbacks = [
    EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=150,
    batch_size=32,
    callbacks=callbacks,
    verbose=2
)

loss, acc = model.evaluate(X_test, y_test)
print(f"Test accuracy: {acc*100:.2f}%")

model.save("gesture_recognition_cnn.h5", save_format="h5")

print("Model saved as gesture_recognition_cnn.h5")

Input shape: (3160, 234, 2)
Training samples: 2528, Test samples: 632


/home/yassin/miniconda3/envs/tf/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_4 (Conv1D)               │ (None, 232, 64)        │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_5 (Conv1D)               │ (None, 230, 64)        │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 230, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 14720)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 128)            │     1,884,288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 8)              │         1,032 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,898,120 (7.24 MB)

 Trainable params: 1,898,120 (7.24 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/150
79/79 - 4s - 53ms/step - accuracy: 0.3394 - loss: 14.1119 - val_accuracy: 0.6930 - val_loss: 1.0623 - learning_rate: 1.0000e-03
Epoch 2/150
79/79 - 0s - 4ms/step - accuracy: 0.5190 - loss: 1.3152 - val_accuracy: 0.7373 - val_loss: 0.8193 - learning_rate: 1.0000e-03
Epoch 3/150
79/79 - 0s - 3ms/step - accuracy: 0.6060 - loss: 1.0700 - val_accuracy: 0.6677 - val_loss: 0.8638 - learning_rate: 1.0000e-03
Epoch 4/150
79/79 - 0s - 3ms/step - accuracy: 0.6491 - loss: 1.0055 - val_accuracy: 0.8307 - val_loss: 0.6410 - learning_rate: 1.0000e-03
Epoch 5/150
79/79 - 0s - 3ms/step - accuracy: 0.7085 - loss: 0.8309 - val_accuracy: 0.8528 - val_loss: 0.4741 - learning_rate: 1.0000e-03
Epoch 6/150
79/79 - 0s - 3ms/step - accuracy: 0.7116 - loss: 0.8432 - val_accuracy: 0.8323 - val_loss: 0.5028 - learning_rate: 1.0000e-03
Epoch 7/150
79/79 - 0s - 3ms/step - accuracy: 0.7540 - loss: 0.7301 - val_accuracy: 0.8133 - val_loss: 0.5010 - learning_rate: 1.0000e-03
Epoch 8/150
79/79 - 0s - 3ms/ste

Test accuracy: 94.78%
Model saved as gesture_recognition_cnn.h5
